# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a complete guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. All dataset entities, including record sets and fields, are referenced by their `@id` per best practice.

### Dataset Source
This dataset is published with a Croissant schema and is accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and explore available record sets using the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'Unknown')}")

## 2. Data Overview

List available record sets and their fields, along with their `@id` attributes.

In [ ]:
# Get all record sets and their IDs
record_sets = dataset.record_sets
print('Available record sets in the dataset:')
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")

# For illustration, list all fields for each record set
print("\nFields available for each RecordSet:")
for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    for field in rs.fields:
        print(f"  - Field: {field.name}, @id: {field.id}, DataType: {getattr(field, 'dataType', 'Unknown')}")

## 3. Data Extraction

Load data from a selected record set into a DataFrame using field `@id`s. Adjust `record_set_id` to select a different record set. All keys reference their `@id` from the dataset.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]

# Load each into a DataFrame (referenced by @id)
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Example: display columns for the main (first listed) record set
example_rs_id = record_set_ids[0]
print(f"Fields (column @ids) in RecordSet '{example_rs_id}':\n", dataframes[example_rs_id].columns.tolist())
dataframes[example_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply basic filtering and statistics on numeric fields. All operations use field `@id`s. Modify `record_set_id`, `numeric_field_id`, and `group_field_id` as needed based on your record set's structure (see above for options).

In [ ]:
# Specify record set and fields by @id
record_set_id = example_rs_id

# Attempt to auto-select a numeric field by data type
numeric_field_id = None
for rs in dataset.record_sets:
    if rs.id == record_set_id:
        for field in rs.fields:
            dtype = getattr(field, 'dataType', '').lower()
            if dtype in ('number', 'float', 'integer'):
                numeric_field_id = field.id
                break
        break
if numeric_field_id is None:
    # Fallback to first column
    numeric_field_id = dataframes[record_set_id].columns[0]
print(f"Using numeric field: {numeric_field_id}")

# EDA: remove missing and non-numeric, filter and normalize
df = dataframes[record_set_id].copy()
df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()]  # Remove NAs
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].mean() if not df[numeric_field_id].empty else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
if not filtered_df[numeric_field_id].empty:
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by a likely categorical field (e.g., the second column)
group_field_id = df.columns[1] if len(df.columns) > 1 else None
if group_field_id is not None:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped.head())

## 5. Visualization

Visualize distributions or relationships in the selected record set. Modify fields referenced by their `@id` if needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=30)
plt.title(f"Distribution of '{numeric_field_id}' in '{record_set_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Grouped boxplot if possible
if group_field_id is not None:
    plt.figure(figsize=(10, 5))
    # Use the original filtered data
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

This notebook provided a step-by-step exploration using the `mlcroissant` library, referencing all entities by their `@id`. Use these techniques to further analyze or prepare the FAIR² dataset for downstream tasks or modeling. For more information about the Croissant format, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/).